
## Collecting Data from API

Opening conda prompt:

1. cd /d D:\crypto_trading

2. conda activate cryptotrading

3. jupyter notebook

https://developers.binance.com/docs/binance-spot-api-docs

https://www.bitmex.com/app/apiOverview

In [21]:
# import necessary files
import logging
from exchanges.binance import BinanceClient
from exchanges.coinbase import CoinbaseClient
from database import Hdf5Client
from utils.time_convertor import *
import pandas as pd
import numpy as np
import time

#strategies
import strategies.obv
import strategies.ichimoku as ic

In [22]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 5000)
pd.set_option('display.expand_frame_repr', False)  # prevent wrapping of DataFrame

In [23]:
h5_db= Hdf5Client("Binance")
data= h5_db.get_data("BTCUSDT", from_time= 0, to_time= int(time.time() * 1000))


In [24]:
#import and format the data
data= resample_timeframe(data, "1h")
data

,open,high,low,close,volume
timestamp,,,,,
2025-03-26 06:00:00,87491.02,87923.92,87491.03,87909.07,411.588490
2025-03-26 07:00:00,87881.90,88275.00,87774.36,88140.45,1053.471940
2025-03-26 08:00:00,88132.30,88200.00,87888.00,87959.73,525.113310
2025-03-26 09:00:00,87959.73,88056.61,87853.99,87939.99,393.097500
2025-03-26 10:00:00,87939.99,88259.81,87940.00,88208.47,556.205410
2025-03-26 11:00:00,88208.46,88242.00,87960.00,87954.96,940.317970
2025-03-26 12:00:00,87952.11,88000.00,87613.88,87813.17,825.399060
2025-03-26 13:00:00,87762.00,87920.08,86856.83,86841.31,1682.405840
2025-03-26 14:00:00,86738.40,87294.05,86473.03,86773.14,1940.957520


## Testing out strategies: 

##### Ichimoku:

In [25]:
ichimoku_df= data.copy()


In [26]:
#tenkan
ichimoku_df["rolling_min_tenkan"]= ichimoku_df['low'].rolling(window=9).min()
ichimoku_df["rolling_max_tenkan"]= ichimoku_df['high'].rolling(window=9).max()
ichimoku_df['tenkan_sen'] = (ichimoku_df["rolling_min_tenkan"]+ ichimoku_df["rolling_max_tenkan"]) / 2

ichimoku_df.drop(columns=["rolling_min_tenkan", "rolling_max_tenkan"], axis=1, inplace=True)

# Base Line: Kijun Sen
ichimoku_df["rolling_min_kijun"]= ichimoku_df['low'].rolling(window=26).min()
ichimoku_df["rolling_max_kijun"]= ichimoku_df['high'].rolling(window=26).max()
ichimoku_df['kijun_sen'] = (ichimoku_df["rolling_min_kijun"]+ichimoku_df["rolling_max_kijun"]) / 2

ichimoku_df.drop(columns=["rolling_min_kijun", "rolling_max_kijun"], axis=1, inplace=True)   

# Leading Span A: Senkou Span A
ichimoku_df['senkou_span_a'] = ((ichimoku_df['tenkan_sen'] + ichimoku_df['kijun_sen']) / 2).shift(26)

# Leading Span B: Senkou Span B
ichimoku_df['rolling_senkou'] = ((ichimoku_df['high'].rolling(window=26*2).max()) + (ichimoku_df['low'].rolling(window=26*2)).min()) / 2
ichimoku_df['senkou_span_b'] = ichimoku_df['rolling_senkou'].shift(26)

ichimoku_df.drop(columns=["rolling_senkou"], axis=1, inplace=True)

# Lagging Span: Chikou Span
# Chikou Span is the closing price shifted back by 26 periods
ichimoku_df['chikou_span'] = ichimoku_df['close'].shift(-1 * 26)

ichimoku_df.dropna(inplace=True)

   

#Signal
ichimoku_df["tenkan_minus_kijun"]= ichimoku_df["tenkan_sen"] - ichimoku_df["kijun_sen"]
ichimoku_df["prev_tenkan_minus_kijun"]= ichimoku_df["tenkan_minus_kijun"].shift(1)
ichimoku_df["prev_senkou_span_a"]= ichimoku_df["senkou_span_a"].shift(1)
ichimoku_df["prev_senkou_span_b"]= ichimoku_df["senkou_span_b"].shift(1)
ichimoku_df["signal"]= np.where(
        (ichimoku_df["tenkan_minus_kijun"] > 0)  #tenkan sen and kijun sen crossover
        & (ichimoku_df["prev_tenkan_minus_kijun"] < 0) 
        & (ichimoku_df["close"]> ichimoku_df["senkou_span_a"]) # close price greater than senkou span A
        & (ichimoku_df["close"] > ichimoku_df["senkou_span_b"]) # close price greater than senkou span B
        & (ichimoku_df["close"]> ichimoku_df["chikou_span"]) # close price greater than chikou span
        #& (ichimoku_df["senkou_span_a"] > ichimoku_df["senkou_span_b"]) # senkou span A greater than senkou span B
        #& (ichimoku_df["prev_senkou_span_a"] < ichimoku_df["prev_senkou_span_b"])
        , 1,  # bullish or long signal

        np.where(
        (ichimoku_df["tenkan_minus_kijun"] < 0) 
        & (ichimoku_df["prev_tenkan_minus_kijun"] > 0) 
        & (ichimoku_df["close"]< ichimoku_df["senkou_span_a"])
        & (ichimoku_df["close"]< ichimoku_df["senkou_span_b"])
        & (ichimoku_df["close"]< ichimoku_df["chikou_span"])
        #& (ichimoku_df["senkou_span_a"] < ichimoku_df["senkou_span_b"])
        #& (ichimoku_df["prev_senkou_span_a"] > ichimoku_df["prev_senkou_span_b"])
        , -1, 0) # bearish or short signal, otherwise no signal
)  

# Calculate PnL
# Dataframe only with 1 or -1 signals
signal_data= ichimoku_df[ichimoku_df["signal"] != 0].copy()
# Calculate the percentage change in close price and shift the signal by 1 to align with the previous signal
signal_data["pnl"]= signal_data["close"].pct_change() * signal_data["signal"].shift(1)

#Calculating Maximum Drawdown
signal_data["cum_pnl"]= signal_data["pnl"].cumsum()
signal_data["max_cum_pnl"]= signal_data["cum_pnl"].cummax()
signal_data["drawdown"]= signal_data["max_cum_pnl"] - signal_data["cum_pnl"]

#Calculating the maximum drwadown in percentage:
signal_data["cum_returns"]= (1 + signal_data["pnl"]).cumprod()
signal_data["max_cum_returns"]= signal_data["cum_returns"].cummax()
signal_data["drawdown_pct"]= (signal_data["cum_returns"] - signal_data["max_cum_returns"]) / signal_data["max_cum_returns"]
signal_data["drawdown_pct"]= signal_data["drawdown_pct"].fillna(0) * 100  # Convert to percentage


In [27]:
signal_data.head(30)

,open,high,low,close,volume,tenkan_sen,kijun_sen,senkou_span_a,senkou_span_b,chikou_span,tenkan_minus_kijun,prev_tenkan_minus_kijun,prev_senkou_span_a,prev_senkou_span_b,signal,pnl,cum_pnl,max_cum_pnl,drawdown,cum_returns,max_cum_returns,drawdown_pct
timestamp,,,,,,,,,,,,,,,,,,,,,,
2025-03-30 19:00:00,82707.83,82908.11,82550.05,82548.81,258.33176,82710.005,82797.170,82975.6475,84696.745,82563.99,-87.165,197.240,83154.4925,84875.590,-1,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
2025-04-15 02:00:00,84983.73,85300.01,84728.76,85232.83,611.38477,84838.995,84808.360,84336.7875,84452.430,83587.99,30.635,-22.815,84301.9200,84452.430,1,-0.032514,-0.032514,-0.032514,0.000000,0.967486,0.967486,0.000000
2025-04-17 02:00:00,84296.46,84337.99,84038.00,84272.86,254.74825,83980.370,84323.005,84489.7475,84777.575,84663.71,-342.635,67.600,84771.0050,84777.575,-1,-0.011263,-0.043777,-0.032514,0.011263,0.956589,0.967486,-1.126291
2025-04-24 00:00:00,93679.95,93787.65,93253.61,93357.22,891.30897,93566.730,93320.755,91188.1925,89204.870,92992.33,245.975,-238.060,90929.6800,89181.385,1,-0.107797,-0.151574,-0.032514,0.119060,0.853472,0.967486,-11.784579
2025-04-28 23:00:00,94809.52,95180.00,94809.52,95011.18,768.18162,94328.100,94227.140,94292.5675,94498.455,94527.73,100.960,-29.040,94264.9025,94591.450,1,0.017716,-0.133858,-0.032514,0.101343,0.868592,0.967486,-10.221713
2025-04-29 22:00:00,94580.50,94735.65,94060.15,94163.67,1003.36560,94760.070,94760.840,94390.1200,94227.140,94405.47,-0.770,73.345,94390.1200,94227.140,-1,-0.008920,-0.142778,-0.032514,0.110264,0.860844,0.967486,-11.022545
2025-04-30 13:00:00,94452.00,94649.50,93074.16,93032.58,2712.01725,94151.305,94267.075,94652.1025,94227.140,96854.30,-115.770,192.950,94604.2375,94227.140,-1,0.012012,-0.130766,-0.032514,0.098252,0.871185,0.967486,-9.953752
2025-05-06 10:00:00,94197.56,94329.63,93881.77,93942.83,952.04682,94259.890,94406.335,94531.9700,95101.405,96966.08,-146.445,75.495,94599.9700,95101.405,-1,-0.009784,-0.140550,-0.032514,0.108036,0.862661,0.967486,-10.834783
2025-05-11 11:00:00,104633.54,104777.00,104391.51,104652.01,611.74101,104061.035,104020.480,103367.9375,101550.950,104319.99,40.555,-13.765,103425.2025,101549.695,1,-0.113997,-0.254547,-0.032514,0.222033,0.764320,0.967486,-20.999331


In [10]:
import plotly.graph_objects as go
import plotly.offline as pyo
fig= go.Figure()

In [13]:
# plot ohlc candles for btcusdt

fig.add_trace(go.Candlestick(
    x= data.index,
    open= data["open"],
    high= data["high"],
    low= data["low"],
    close= data["close"],
    name= "BTCUSDT OHLC"))

#Ichimoku Lines
fig.add_trace(go.Scatter(x=data.index, y=data['tenkan_sen'], line=dict(color='blue'), name='Tenkan Sen'))
fig.add_trace(go.Scatter(x=data.index, y=data['kijun_sen'], line=dict(color='purple'), name='Kijun Sen'))
fig.add_trace(go.Scatter(x=data.index, y=data['senkou_span_a'], line=dict(color='green'), name='Senkou Span A', fill=None))
fig.add_trace(go.Scatter(x=data.index, y=data['senkou_span_b'], line=dict(color='red'), name='Senkou Span B', fill='tonexty', fillcolor='rgba(50,200,50,0.2)'))
fig.add_trace(go.Scatter(x=data.index, y=data['chikou_span'], line=dict(color='orange'), name='Chikou Span'))

# Mark Buy Signals
buy_signals = data[data['signal'] == 1]
fig.add_trace(go.Scatter(mode='markers', x=buy_signals.index, y=buy_signals['close'],
                         marker=dict(color='green', size=10, symbol='triangle-up'),
                         name='Buy Signal'))

# Mark Sell Signals
sell_signals = data[data['signal'] == -1]
fig.add_trace(go.Scatter(mode='markers', x=sell_signals.index, y=sell_signals['close'],
                         marker=dict(color='red', size=10, symbol='triangle-down'),
                         name='Sell Signal'))


# Update Layout
fig.update_layout(
    title="Ichimoku Trading Strategy Visualization",
    xaxis_title="Date",
    yaxis_title="Price",
    legend_title="Legend",
    width=1200,
    height=600
)

pyo.plot(fig, filename= "ichimoku.html")

'ichimoku.html'